# NRC-VAD Valence Sentiment

This notebook estimates annual connotational sentiment for ADHD, Autism, and the three baseline terms. It keeps the Baes-style local-collocate index already implemented here, but now joins the locked frame-classifier output so target estimates are substantive-frame aware.


## Setup

The diachronic axis is publication year (`lsc_year`). ADHD and Autism target contexts are restricted to substantive core discourse for semantic estimates: `clinical_only`, `lived_only`, and `mixed`. Baselines are not frame-labelled and remain separate comparator series.


In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import hashlib
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import spacy

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#263238",
        "axes.labelcolor": "#263238",
        "axes.titlecolor": "#263238",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#D7DEE2",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.7,
        "font.family": "DejaVu Sans",
        "font.size": 10.5,
        "legend.frameon": False,
        "xtick.color": "#263238",
        "ytick.color": "#263238",
    }
)


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/contexts/lsc_mention_contexts.parquet"
FRAME_LABEL_PATH = PROJECT_ROOT / "data/processed/lsc/classification/lsc_target_context_frame_labels.csv"
NRC_VAD_PATH = PROJECT_ROOT / "data/external/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt"
INTERIM_VAD_DIR = PROJECT_ROOT / "data/interim/lsc/vad"
SENTIMENT_DIR = PROJECT_ROOT / "data/processed/lsc/sentiment"
FIGURE_DIR = PROJECT_ROOT / "reports/figures/lsc/sentiment"

INTERIM_VAD_DIR.mkdir(parents=True, exist_ok=True)
SENTIMENT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

VAD_MATCHES_PATH = INTERIM_VAD_DIR / "lsc_vad_collocate_matches.parquet"
VAD_CONTEXT_COVERAGE_PATH = INTERIM_VAD_DIR / "lsc_vad_context_coverage.parquet"
ANNUAL_VALENCE_PATH = SENTIMENT_DIR / "lsc_sentiment_annual_valence.csv"
COVERAGE_PATH = SENTIMENT_DIR / "lsc_sentiment_coverage.csv"
TOP_COLLOCATES_PATH = SENTIMENT_DIR / "lsc_sentiment_top_collocates.csv"
AUDIT_FLAGS_PATH = SENTIMENT_DIR / "lsc_sentiment_audit_flags.csv"
TREND_SUMMARY_PATH = SENTIMENT_DIR / "lsc_sentiment_trend_models.csv"
FRAME_CONTEXT_DIAGNOSTICS_PATH = SENTIMENT_DIR / "lsc_sentiment_frame_context_diagnostics.csv"
VALENCE_PLOT_PATH = FIGURE_DIR / "lsc_sentiment_valence_trajectories.png"

EXPECTED_UNITS = ["ADHD", "Autism", "frustration", "loneliness", "sadness"]
TARGET_UNITS = ["ADHD", "Autism"]
BASELINE_UNITS = ["frustration", "loneliness", "sadness"]
EXPECTED_YEARS = list(range(2014, 2027))
CORE_TARGET_FRAMES = ["clinical_only", "lived_only", "mixed"]
TARGET_FRAME_STRATA = ["substantive_core_overall", *CORE_TARGET_FRAMES]
BASELINE_FRAME_STRATUM = "unframed_baseline"
BOOTSTRAP_REPETITIONS = 500
BOOTSTRAP_SEED = 123
LOW_MATCHED_TOKEN_COVERAGE_WARN = 0.35
LOW_CONTEXT_COVERAGE_WARN = 0.50
TOP_COLLOCATE_SHARE_WARN = 0.20
MIN_FRAME_CONTEXTS_FOR_INTERPRETATION = 100
MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION = 50
DW_AUTOCORRELATION_LOW = 1.25
DW_AUTOCORRELATION_HIGH = 2.75

LSC_UNIT_LABELS = {
    "ADHD": "ADHD",
    "Autism": "Autism",
    "frustration": "Frustration",
    "loneliness": "Loneliness",
    "sadness": "Sadness",
}
LSC_UNIT_COLORS = {
    "ADHD": "#2F6F9F",
    "Autism": "#B66A4A",
    "frustration": "#4F8F78",
    "loneliness": "#7FA68A",
    "sadness": "#9AA6A1",
}
LSC_UNIT_MARKERS = {
    "ADHD": "o",
    "Autism": "s",
    "frustration": "^",
    "loneliness": "D",
    "sadness": "v",
}
FRAME_LABELS = {
    "substantive_core_overall": "Overall",
    "clinical_only": "Clinical/disorder framing",
    "lived_only": "Lived-experience framing",
    "mixed": "Mixed clinical/lived framing",
    "unframed_baseline": "Comparator term",
}
FRAME_COLORS = {
    "substantive_core_overall": "#263238",
    "clinical_only": "#4F8DB3",
    "lived_only": "#C98263",
    "mixed": "#79A889",
    "substantive_other": "#A998C9",
    "non_substantive_or_insufficient": "#B8C0C5",
    "unframed_baseline": "#7B8785",
}
CONDITION_FRAME_COLORS = {
    "ADHD": {
        "substantive_core_overall": "#2F6F9F",
        "clinical_only": "#75A9C8",
        "lived_only": "#AECFE0",
        "mixed": "#D4E4EC",
    },
    "Autism": {
        "substantive_core_overall": "#B66A4A",
        "clinical_only": "#CE8D70",
        "lived_only": "#E1B49D",
        "mixed": "#F2D8CF",
    },
}
FRAME_MARKERS = {
    "substantive_core_overall": "o",
    "clinical_only": "s",
    "lived_only": "^",
    "mixed": "D",
    "unframed_baseline": "o",
}
LSC_FIGURE_DPI = 300

assert CONTEXT_PATH.exists(), CONTEXT_PATH
assert FRAME_LABEL_PATH.exists(), FRAME_LABEL_PATH
assert NRC_VAD_PATH.exists(), NRC_VAD_PATH

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError as exc:
    raise OSError(
        "Missing spaCy model en_core_web_sm. Install it with: python -m spacy download en_core_web_sm"
    ) from exc

PROJECT_ROOT


PosixPath('/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak')

## Load Inputs And Join Frame Labels

The stable `context_id` is reconstructed from the same fields used by the classifier application notebook. Target contexts in the three core substantive frames are duplicated into their hard frame stratum and the `substantive_core_overall` aggregate. Non-substantive and substantive-other target contexts are retained only in the diagnostics table.


In [2]:
context_columns = [
    "doc_id",
    "lsc_year",
    "published_year",
    "source_year",
    "analysis_unit",
    "term_role",
    "target_group",
    "raw_form",
    "matched_text",
    "mention_start_char",
    "mention_end_char",
    "collapsed_raw_forms",
    "collapsed_matched_texts",
    "registered_domain",
    "token_window_5",
]
contexts = pd.read_parquet(CONTEXT_PATH, columns=context_columns).reset_index(drop=True)
contexts["source_context_row_id"] = contexts.index.astype(int)

observed_units = sorted(contexts["analysis_unit"].dropna().unique())
observed_years = sorted(contexts["lsc_year"].dropna().astype(int).unique())
missing_units = sorted(set(EXPECTED_UNITS) - set(observed_units))
missing_years = sorted(set(EXPECTED_YEARS) - set(observed_years))
if missing_units:
    raise RuntimeError(f"Missing expected analysis units: {missing_units}")
if missing_years:
    raise RuntimeError(f"Missing expected LSC years: {missing_years}")
if contexts["token_window_5"].fillna("").str.strip().eq("").any():
    raise RuntimeError("Some context rows have empty token_window_5 values.")


def stable_context_id(row: pd.Series) -> str:
    value = "|".join(
        str(row.get(column, ""))
        for column in ["doc_id", "analysis_unit", "raw_form", "mention_start_char", "mention_end_char"]
    )
    return hashlib.sha1(value.encode("utf-8")).hexdigest()[:16]


label_columns = [
    "context_id",
    "predicted_derived_frame",
    "p_substantive",
    "p_clinical_given_substantive",
    "p_lived_given_substantive",
]
frame_labels = pd.read_csv(FRAME_LABEL_PATH, usecols=label_columns)
if frame_labels["context_id"].duplicated().any():
    raise RuntimeError("Frame-label handoff contains duplicate context_id values.")

target_mask = contexts["analysis_unit"].isin(TARGET_UNITS)
contexts.loc[target_mask, "context_id"] = contexts.loc[target_mask].apply(stable_context_id, axis=1)
contexts = contexts.merge(frame_labels, on="context_id", how="left")
missing_target_labels = contexts.loc[target_mask, "predicted_derived_frame"].isna().sum()
if missing_target_labels:
    raise RuntimeError(f"Missing frame labels for {missing_target_labels:,} target contexts.")

frame_context_diagnostics = (
    contexts.loc[target_mask]
    .groupby(["analysis_unit", "lsc_year", "predicted_derived_frame"], as_index=False)
    .agg(contexts=("source_context_row_id", "size"), documents=("doc_id", "nunique"))
)
frame_context_diagnostics["included_in_semantic_estimates"] = frame_context_diagnostics[
    "predicted_derived_frame"
].isin(CORE_TARGET_FRAMES)
frame_context_diagnostics["small_cell_flag"] = frame_context_diagnostics[
    "included_in_semantic_estimates"
] & (
    frame_context_diagnostics["contexts"].lt(MIN_FRAME_CONTEXTS_FOR_INTERPRETATION)
    | frame_context_diagnostics["documents"].lt(MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION)
)
frame_context_diagnostics.to_csv(FRAME_CONTEXT_DIAGNOSTICS_PATH, index=False)

baseline_contexts = contexts.loc[~target_mask].copy()
baseline_contexts["frame_stratum"] = BASELINE_FRAME_STRATUM

core_target_contexts = contexts.loc[
    target_mask & contexts["predicted_derived_frame"].isin(CORE_TARGET_FRAMES)
].copy()
target_by_frame = core_target_contexts.copy()
target_by_frame["frame_stratum"] = target_by_frame["predicted_derived_frame"]
target_overall = core_target_contexts.copy()
target_overall["frame_stratum"] = "substantive_core_overall"

analysis_contexts = pd.concat([baseline_contexts, target_overall, target_by_frame], ignore_index=True, sort=False)
analysis_contexts["context_row_id"] = np.arange(len(analysis_contexts), dtype=int)

vad = pd.read_csv(NRC_VAD_PATH, sep="\t")
expected_vad_columns = {"term", "valence", "arousal", "dominance"}
if set(vad.columns) != expected_vad_columns:
    raise RuntimeError(f"Unexpected NRC-VAD columns: {vad.columns.tolist()}")
for column in ["valence", "arousal", "dominance"]:
    if not vad[column].between(-1, 1).all():
        raise RuntimeError(f"NRC-VAD {column} values outside expected [-1, 1] range.")

analysis_context_summary = (
    analysis_contexts.groupby(["analysis_unit", "frame_stratum"], as_index=False)
    .agg(contexts=("context_row_id", "size"), documents=("doc_id", "nunique"))
    .sort_values(["analysis_unit", "frame_stratum"])
)
print(f"Original contexts: {len(contexts):,}")
print(f"Analysis context rows after frame expansion: {len(analysis_contexts):,}")
print(f"NRC-VAD terms: {len(vad):,}")
analysis_context_summary


Original contexts: 293,670
Analysis context rows after frame expansion: 311,030
NRC-VAD terms: 54,801


,analysis_unit,frame_stratum,contexts,documents
0,ADHD,clinical_only,12039,8006
1,ADHD,lived_only,3610,2730
2,ADHD,mixed,2000,1634
3,ADHD,substantive_core_overall,17649,11444
4,Autism,clinical_only,20306,12959
5,Autism,lived_only,12826,9068
6,Autism,mixed,6331,4735
7,Autism,substantive_core_overall,39463,24141
8,frustration,unframed_baseline,105482,93663
9,loneliness,unframed_baseline,38052,30431


## Prepare NRC-VAD Lookup

NRC-VAD v2.1 includes unigrams and multi-word expressions. To use lemmatised context windows consistently, lexicon terms are also tokenised and lemmatised with the same spaCy model. If multiple surface entries collapse to the same lemma phrase, their VAD scores are averaged.


In [3]:
lexicon_records: list[dict[str, object]] = []
vad_terms = vad["term"].fillna("").map(str).tolist()
for doc, row in zip(nlp.pipe(vad_terms, batch_size=1000), vad.itertuples(index=False)):
    lemma_tokens = [token.lemma_.lower() for token in doc if token.is_alpha and len(token.lemma_) > 1]
    if not lemma_tokens:
        continue
    lexicon_records.append(
        {
            "lexicon_key": " ".join(lemma_tokens),
            "lexicon_tuple": tuple(lemma_tokens),
            "lexicon_length": len(lemma_tokens),
            "term": row.term,
            "valence": float(row.valence),
            "arousal": float(row.arousal),
            "dominance": float(row.dominance),
        }
    )

lexicon = pd.DataFrame(lexicon_records)
lexicon_lookup = (
    lexicon.groupby(["lexicon_key", "lexicon_length"], as_index=False)
    .agg(
        valence=("valence", "mean"),
        arousal=("arousal", "mean"),
        dominance=("dominance", "mean"),
        source_terms=("term", lambda values: " | ".join(sorted(set(values))[:5])),
        source_term_count=("term", "nunique"),
    )
)
lexicon_lookup["lexicon_tuple"] = lexicon_lookup["lexicon_key"].str.split().map(tuple)

unigram_lookup = {
    row.lexicon_tuple: row
    for row in lexicon_lookup.loc[lexicon_lookup["lexicon_length"] == 1].itertuples(index=False)
}
mwe_lookup = {
    row.lexicon_tuple: row
    for row in lexicon_lookup.loc[lexicon_lookup["lexicon_length"] > 1].itertuples(index=False)
}
mwe_lengths = sorted({len(key) for key in mwe_lookup}, reverse=True)

print(f"Normalised NRC-VAD keys: {len(lexicon_lookup):,}")
print(f"Unigram keys: {len(unigram_lookup):,}")
print(f"MWE keys: {len(mwe_lookup):,}")
lexicon_lookup.head()


Normalised NRC-VAD keys: 50,815
Unigram keys: 41,110
MWE keys: 9,705


,lexicon_key,lexicon_length,valence,arousal,dominance,source_terms,source_term_count,lexicon_tuple
0,aaaaaaah,1,-0.042,0.212,-0.418,aaaaaaah,1,"(aaaaaaah,)"
1,aaaah,1,0.040,0.272,-0.436,aaaah,1,"(aaaah,)"
2,aardvark,1,-0.146,-0.020,-0.126,aardvark,1,"(aardvark,)"
3,aback,1,-0.230,-0.186,-0.424,aback,1,"(aback,)"
4,abacus,1,0.020,-0.448,-0.030,abacus,1,"(abacus,)"


## Extract VAD-Matched Collocates

The collocate extraction remains the original Baes-style local-window procedure: remove only the focal mention's lexical material, keep stopwords, and greedily match NRC-VAD multi-word expressions before unigrams. The saved VAD handoff now includes `frame_stratum` so the Intensity notebook can reuse exactly the same frame-aware preprocessing.


In [4]:
TARGET_RAW_FORM_EXCLUSION_TOKENS = {
    "adhd": {"adhd"},
    "attention_deficit": {"attention", "deficit", "hyperactivity", "disorder"},
    "autism": {"autism"},
    "autistic": {"autistic"},
    "autism_spectrum": {"autism", "spectrum"},
    "asd_disambiguated": {"asd"},
}
BASELINE_RAW_FORM_EXCLUSION_TOKENS = {
    "frustration": {"frustration"},
    "sadness": {"sadness"},
    "loneliness": {"loneliness"},
}
RAW_FORM_EXCLUSION_TOKENS_BY_ROLE = {
    "target": TARGET_RAW_FORM_EXCLUSION_TOKENS,
    "baseline": BASELINE_RAW_FORM_EXCLUSION_TOKENS,
}


def split_pipe_values(value: object) -> list[str]:
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("|") if part.strip()]


def exclusion_tokens_for_row(row: pd.Series) -> set[str]:
    raw_forms = {str(row["raw_form"])} | set(split_pipe_values(row.get("collapsed_raw_forms")))
    role_exclusions = RAW_FORM_EXCLUSION_TOKENS_BY_ROLE.get(str(row["term_role"]), {})
    tokens: set[str] = set()
    for raw_form in raw_forms:
        tokens.update(role_exclusions.get(raw_form, {raw_form.replace("_", " ")}))
    matched_text = str(row.get("matched_text") or "")
    if matched_text:
        tokens.update(part.lower() for part in re.findall(r"[A-Za-z]+", matched_text))
    return tokens


def lexical_tokens(doc, excluded_terms: set[str]) -> list[dict[str, object]]:
    tokens = []
    for token in doc:
        lemma = token.lemma_.lower()
        lower = token.text.lower()
        if not token.is_alpha or token.like_num or len(lemma) <= 1:
            continue
        if lemma in excluded_terms or lower in excluded_terms:
            continue
        tokens.append({"token_index": token.i, "text": token.text, "lemma": lemma})
    return tokens


def match_vad_units(tokens: list[dict[str, object]]) -> list[dict[str, object]]:
    matches: list[dict[str, object]] = []
    index = 0
    while index < len(tokens):
        matched = None
        for length in mwe_lengths:
            if index + length > len(tokens):
                continue
            key = tuple(token["lemma"] for token in tokens[index : index + length])
            if key in mwe_lookup:
                matched = (length, mwe_lookup[key], "mwe")
                break
        if matched is None:
            key = (tokens[index]["lemma"],)
            if key in unigram_lookup:
                matched = (1, unigram_lookup[key], "unigram")
        if matched is None:
            index += 1
            continue
        length, lexicon_row, collocate_type = matched
        span_tokens = tokens[index : index + length]
        matches.append(
            {
                "collocate": lexicon_row.lexicon_key,
                "collocate_type": collocate_type,
                "surface_text": " ".join(str(token["text"]) for token in span_tokens),
                "token_start_in_window": int(span_tokens[0]["token_index"]),
                "token_end_in_window": int(span_tokens[-1]["token_index"] + 1),
                "matched_token_count": int(length),
                "valence": float(lexicon_row.valence),
                "arousal": float(lexicon_row.arousal),
                "dominance": float(lexicon_row.dominance),
                "source_terms": lexicon_row.source_terms,
                "source_term_count": int(lexicon_row.source_term_count),
            }
        )
        index += length
    return matches


match_rows: list[dict[str, object]] = []
coverage_rows: list[dict[str, object]] = []

for doc, (_, row) in zip(
    nlp.pipe(analysis_contexts["token_window_5"].astype(str), batch_size=1000),
    analysis_contexts.iterrows(),
):
    excluded_terms = exclusion_tokens_for_row(row)
    tokens = lexical_tokens(doc, excluded_terms)
    row_matches = match_vad_units(tokens)
    matched_token_positions = sum(match["matched_token_count"] for match in row_matches)

    coverage_rows.append(
        {
            "context_row_id": int(row["context_row_id"]),
            "source_context_row_id": int(row["source_context_row_id"]),
            "context_id": row.get("context_id"),
            "doc_id": row["doc_id"],
            "lsc_year": int(row["lsc_year"]),
            "analysis_unit": row["analysis_unit"],
            "term_role": row["term_role"],
            "target_group": row["target_group"],
            "raw_form": row["raw_form"],
            "frame_stratum": row["frame_stratum"],
            "predicted_derived_frame": row.get("predicted_derived_frame"),
            "candidate_collocate_tokens": len(tokens),
            "matched_vad_units": len(row_matches),
            "matched_token_positions": int(matched_token_positions),
            "has_vad_match": bool(row_matches),
        }
    )

    for match in row_matches:
        match_rows.append(
            {
                "context_row_id": int(row["context_row_id"]),
                "source_context_row_id": int(row["source_context_row_id"]),
                "context_id": row.get("context_id"),
                "doc_id": row["doc_id"],
                "lsc_year": int(row["lsc_year"]),
                "published_year": int(row["published_year"]),
                "source_year": int(row["source_year"]),
                "analysis_unit": row["analysis_unit"],
                "term_role": row["term_role"],
                "target_group": row["target_group"],
                "raw_form": row["raw_form"],
                "frame_stratum": row["frame_stratum"],
                "predicted_derived_frame": row.get("predicted_derived_frame"),
                "registered_domain": row["registered_domain"],
                **match,
            }
        )

vad_matches = pd.DataFrame(match_rows)
context_coverage = pd.DataFrame(coverage_rows)
if vad_matches.empty:
    raise RuntimeError("No NRC-VAD collocates matched the frame-aware LSC contexts.")

vad_matches.to_parquet(VAD_MATCHES_PATH, index=False)
context_coverage.to_parquet(VAD_CONTEXT_COVERAGE_PATH, index=False)

print(f"VAD match rows: {len(vad_matches):,}")
print(f"Contexts with any VAD match: {context_coverage['has_vad_match'].sum():,} / {len(context_coverage):,}")
vad_matches.head()


VAD match rows: 1,972,151
Contexts with any VAD match: 310,688 / 311,030


,context_row_id,source_context_row_id,context_id,doc_id,lsc_year,published_year,source_year,analysis_unit,term_role,target_group,raw_form,frame_stratum,predicted_derived_frame,registered_domain,collocate,collocate_type,surface_text,token_start_in_window,token_end_in_window,matched_token_count,valence,arousal,dominance,source_terms,source_term_count
0,0,11401,NaN,0004cb4b97b156b2,2014,2014,2020,frustration,baseline,baseline,frustration,unframed_baseline,NaN,thegiconnection.com,of,unigram,of,1,2,1,-0.111,-0.119000,-0.161,of | of a,2
1,0,11401,NaN,0004cb4b97b156b2,2014,2014,2020,frustration,baseline,baseline,frustration,unframed_baseline,NaN,thegiconnection.com,continue,unigram,continued,2,3,1,0.288,-0.088667,0.088,continue | continued | continuing,3
2,0,11401,NaN,0004cb4b97b156b2,2014,2014,2020,frustration,baseline,baseline,frustration,unframed_baseline,NaN,thegiconnection.com,failure,unigram,failure,3,4,1,-0.666,0.150000,-0.722,failure,1
3,0,11401,NaN,0004cb4b97b156b2,2014,2014,2020,frustration,baseline,baseline,frustration,unframed_baseline,NaN,thegiconnection.com,and,unigram,and,4,5,1,0.000,0.000000,0.000,and,1
4,0,11401,NaN,0004cb4b97b156b2,2014,2014,2020,frustration,baseline,baseline,frustration,unframed_baseline,NaN,thegiconnection.com,and,unigram,and,6,7,1,0.000,0.000000,0.000,and,1


## Annual Valence Index

Annual valence is the weighted mean of matched NRC-VAD collocate occurrences for each analysis unit, publication year, and frame stratum. Coverage is reported alongside the index so sparse or unstable strata are not over-interpreted.


In [5]:
GROUP_COLUMNS = ["lsc_year", "analysis_unit", "frame_stratum"]

coverage = (
    context_coverage.groupby(GROUP_COLUMNS, as_index=False)
    .agg(
        context_rows=("context_row_id", "size"),
        documents=("doc_id", "nunique"),
        candidate_collocate_tokens=("candidate_collocate_tokens", "sum"),
        matched_vad_units_coverage=("matched_vad_units", "sum"),
        matched_token_positions=("matched_token_positions", "sum"),
        contexts_with_vad_match=("has_vad_match", "sum"),
    )
)
coverage["matched_token_coverage"] = coverage["matched_token_positions"] / coverage["candidate_collocate_tokens"].replace(0, np.nan)
coverage["context_match_coverage"] = coverage["contexts_with_vad_match"] / coverage["context_rows"].replace(0, np.nan)
coverage["small_cell_flag"] = coverage["frame_stratum"].isin(CORE_TARGET_FRAMES) & (
    coverage["context_rows"].lt(MIN_FRAME_CONTEXTS_FOR_INTERPRETATION)
    | coverage["documents"].lt(MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION)
)

annual_valence = (
    vad_matches.groupby([*GROUP_COLUMNS, "term_role", "target_group"], as_index=False)
    .agg(
        valence_mean=("valence", "mean"),
        valence_sd=("valence", "std"),
        arousal_mean_for_reuse=("arousal", "mean"),
        dominance_mean_for_reuse=("dominance", "mean"),
        matched_vad_units=("valence", "size"),
        unique_collocates=("collocate", "nunique"),
        documents_with_matches=("doc_id", "nunique"),
    )
)
annual_valence = annual_valence.merge(coverage, on=GROUP_COLUMNS, how="left")
annual_valence = annual_valence.sort_values(["analysis_unit", "frame_stratum", "lsc_year"]).reset_index(drop=True)
annual_valence.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,valence_mean,valence_sd,arousal_mean_for_reuse,dominance_mean_for_reuse,matched_vad_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_vad_units_coverage,matched_token_positions,contexts_with_vad_match,matched_token_coverage,context_match_coverage,small_cell_flag
0,2014,ADHD,clinical_only,target,ADHD,0.060713,0.389063,-0.027904,0.015758,7410,1406,807,1286,808,9069,7410,7935,1273,0.874959,0.989891,False
1,2015,ADHD,clinical_only,target,ADHD,0.066554,0.384961,-0.032178,0.015869,6765,1379,771,1203,774,8360,6765,7231,1191,0.864952,0.990025,False
2,2016,ADHD,clinical_only,target,ADHD,0.044939,0.403335,-0.021915,0.008918,6532,1357,789,1149,790,8030,6532,7025,1141,0.874844,0.993037,False
3,2017,ADHD,clinical_only,target,ADHD,0.048403,0.388145,-0.018393,0.014013,6178,1346,748,1099,751,7593,6178,6602,1087,0.869485,0.989081,False
4,2018,ADHD,clinical_only,target,ADHD,0.048075,0.398098,-0.020456,0.003309,6869,1425,785,1182,787,8408,6869,7339,1176,0.872859,0.994924,False
5,2019,ADHD,clinical_only,target,ADHD,0.029206,0.402413,-0.013096,0.000834,5789,1218,672,1000,674,7120,5789,6196,995,0.870225,0.995000,False
6,2020,ADHD,clinical_only,target,ADHD,0.044193,0.385056,-0.022576,0.007805,5421,1179,665,959,668,6722,5421,5833,947,0.867748,0.987487,False
7,2021,ADHD,clinical_only,target,ADHD,0.037715,0.405873,-0.015004,0.001729,5281,1252,635,928,636,6430,5281,5639,922,0.876983,0.993534,False
8,2022,ADHD,clinical_only,target,ADHD,0.057969,0.392356,-0.027568,0.021404,5392,1212,633,930,634,6550,5392,5797,923,0.885038,0.992473,False
9,2023,ADHD,clinical_only,target,ADHD,0.053057,0.390987,-0.011918,0.023341,4241,1053,474,750,477,5142,4241,4514,736,0.877869,0.981333,False


## Bootstrap Confidence Intervals

Uncertainty is estimated by resampling documents within each analysis-unit/year/frame stratum. This preserves the existing document-level bootstrap contract while making the frame-specific estimates inspectable.


In [6]:
rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows: list[dict[str, object]] = []

doc_scores = (
    vad_matches.groupby([*GROUP_COLUMNS, "doc_id"], as_index=False)
    .agg(valence_sum=("valence", "sum"), matched_vad_units=("valence", "size"))
)

for group_values, doc_frame in doc_scores.groupby(GROUP_COLUMNS, sort=True):
    valence_sums = doc_frame["valence_sum"].to_numpy(dtype=float)
    unit_counts = doc_frame["matched_vad_units"].to_numpy(dtype=float)
    n_docs = len(doc_frame)
    estimates = np.empty(BOOTSTRAP_REPETITIONS, dtype=float)
    for repetition in range(BOOTSTRAP_REPETITIONS):
        sample_index = rng.integers(0, n_docs, size=n_docs)
        denominator = unit_counts[sample_index].sum()
        estimates[repetition] = valence_sums[sample_index].sum() / denominator if denominator else np.nan
    bootstrap_rows.append(
        {
            "lsc_year": int(group_values[0]),
            "analysis_unit": group_values[1],
            "frame_stratum": group_values[2],
            "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
            "bootstrap_unit": "doc_id",
            "valence_bootstrap_mean": float(np.nanmean(estimates)),
            "valence_ci_low": float(np.nanpercentile(estimates, 2.5)),
            "valence_ci_high": float(np.nanpercentile(estimates, 97.5)),
        }
    )

bootstrap = pd.DataFrame(bootstrap_rows)
annual_valence = annual_valence.merge(bootstrap, on=GROUP_COLUMNS, how="left")
annual_valence.to_csv(ANNUAL_VALENCE_PATH, index=False)
coverage.to_csv(COVERAGE_PATH, index=False)
annual_valence.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,valence_mean,valence_sd,arousal_mean_for_reuse,dominance_mean_for_reuse,matched_vad_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_vad_units_coverage,matched_token_positions,contexts_with_vad_match,matched_token_coverage,context_match_coverage,small_cell_flag,bootstrap_repetitions,bootstrap_unit,valence_bootstrap_mean,valence_ci_low,valence_ci_high
0,2014,ADHD,clinical_only,target,ADHD,0.060713,0.389063,-0.027904,0.015758,7410,1406,807,1286,808,9069,7410,7935,1273,0.874959,0.989891,False,500,doc_id,0.060755,0.049950,0.070766
1,2015,ADHD,clinical_only,target,ADHD,0.066554,0.384961,-0.032178,0.015869,6765,1379,771,1203,774,8360,6765,7231,1191,0.864952,0.990025,False,500,doc_id,0.066259,0.055291,0.077154
2,2016,ADHD,clinical_only,target,ADHD,0.044939,0.403335,-0.021915,0.008918,6532,1357,789,1149,790,8030,6532,7025,1141,0.874844,0.993037,False,500,doc_id,0.045444,0.033434,0.058820
3,2017,ADHD,clinical_only,target,ADHD,0.048403,0.388145,-0.018393,0.014013,6178,1346,748,1099,751,7593,6178,6602,1087,0.869485,0.989081,False,500,doc_id,0.048729,0.036689,0.060673
4,2018,ADHD,clinical_only,target,ADHD,0.048075,0.398098,-0.020456,0.003309,6869,1425,785,1182,787,8408,6869,7339,1176,0.872859,0.994924,False,500,doc_id,0.047781,0.035405,0.060774
5,2019,ADHD,clinical_only,target,ADHD,0.029206,0.402413,-0.013096,0.000834,5789,1218,672,1000,674,7120,5789,6196,995,0.870225,0.995000,False,500,doc_id,0.029004,0.015113,0.041883
6,2020,ADHD,clinical_only,target,ADHD,0.044193,0.385056,-0.022576,0.007805,5421,1179,665,959,668,6722,5421,5833,947,0.867748,0.987487,False,500,doc_id,0.044757,0.031766,0.057799
7,2021,ADHD,clinical_only,target,ADHD,0.037715,0.405873,-0.015004,0.001729,5281,1252,635,928,636,6430,5281,5639,922,0.876983,0.993534,False,500,doc_id,0.037946,0.023391,0.051808
8,2022,ADHD,clinical_only,target,ADHD,0.057969,0.392356,-0.027568,0.021404,5392,1212,633,930,634,6550,5392,5797,923,0.885038,0.992473,False,500,doc_id,0.057535,0.045337,0.069598
9,2023,ADHD,clinical_only,target,ADHD,0.053057,0.390987,-0.011918,0.023341,4241,1053,474,750,477,5142,4241,4514,736,0.877869,0.981333,False,500,doc_id,0.053026,0.037784,0.068572


## Trend Models

Following Baes et al.'s analytical strategy, each reported annual series gets a compact trend model. OLS on centred year is the main descriptive summary; an AR(1)-transformed sensitivity slope is reported only when the residual Durbin-Watson diagnostic is flagged. Quadratic fit is retained as a diagnostic rather than a default model.


In [7]:
def fit_trend(frame: pd.DataFrame, value_column: str) -> dict[str, object]:
    data = frame[["lsc_year", value_column]].dropna().sort_values("lsc_year")
    if len(data) < 3 or data[value_column].nunique() < 2:
        return {
            "n_years": len(data),
            "year_center": np.nan,
            "linear_intercept": np.nan,
            "linear_slope_per_year": np.nan,
            "linear_slope_se": np.nan,
            "linear_p_value": np.nan,
            "linear_r_squared": np.nan,
            "linear_adj_r_squared": np.nan,
            "standardized_beta_year": np.nan,
            "durbin_watson": np.nan,
            "lag1_residual_autocorrelation": np.nan,
            "autocorrelation_flag": False,
            "ar1_sensitivity_slope_per_year": np.nan,
            "ar1_sensitivity_p_value": np.nan,
            "quadratic_adj_r_squared": np.nan,
            "quadratic_delta_adj_r_squared": np.nan,
        }
    years = data["lsc_year"].to_numpy(dtype=float)
    values = data[value_column].to_numpy(dtype=float)
    year_center = float(years.mean())
    x = years - year_center
    result = stats.linregress(x, values)
    fitted = result.intercept + result.slope * x
    residuals = values - fitted
    sse = float(np.sum(residuals**2))
    sst = float(np.sum((values - values.mean()) ** 2))
    r_squared = 1 - sse / sst if sst else np.nan
    n = len(values)
    adj_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - 2) if n > 2 and not np.isnan(r_squared) else np.nan
    std_beta = result.slope * np.std(x, ddof=1) / np.std(values, ddof=1) if np.std(values, ddof=1) else np.nan
    dw_denominator = float(np.sum(residuals**2))
    durbin_watson = float(np.sum(np.diff(residuals) ** 2) / dw_denominator) if dw_denominator else np.nan
    lag1 = float(np.corrcoef(residuals[:-1], residuals[1:])[0, 1]) if n >= 4 and np.std(residuals) else np.nan
    autocorrelation_flag = bool(
        not np.isnan(durbin_watson)
        and (durbin_watson < DW_AUTOCORRELATION_LOW or durbin_watson > DW_AUTOCORRELATION_HIGH)
    )
    ar1_slope = np.nan
    ar1_p_value = np.nan
    if autocorrelation_flag and n >= 5 and not np.isnan(lag1) and abs(lag1) < 0.98:
        y_star = values[1:] - lag1 * values[:-1]
        x_star = x[1:] - lag1 * x[:-1]
        ar1_result = stats.linregress(x_star, y_star)
        ar1_slope = float(ar1_result.slope)
        ar1_p_value = float(ar1_result.pvalue)
    quadratic_adj_r_squared = np.nan
    quadratic_delta = np.nan
    if n >= 6:
        q_coefficients = np.polyfit(x, values, deg=2)
        q_fitted = np.polyval(q_coefficients, x)
        q_sse = float(np.sum((values - q_fitted) ** 2))
        q_r_squared = 1 - q_sse / sst if sst else np.nan
        quadratic_adj_r_squared = 1 - (1 - q_r_squared) * (n - 1) / (n - 3) if n > 3 and not np.isnan(q_r_squared) else np.nan
        quadratic_delta = quadratic_adj_r_squared - adj_r_squared if not np.isnan(quadratic_adj_r_squared) else np.nan
    return {
        "n_years": n,
        "year_center": year_center,
        "linear_intercept": float(result.intercept),
        "linear_slope_per_year": float(result.slope),
        "linear_slope_se": float(result.stderr) if result.stderr is not None else np.nan,
        "linear_p_value": float(result.pvalue),
        "linear_r_squared": float(r_squared),
        "linear_adj_r_squared": float(adj_r_squared),
        "standardized_beta_year": float(std_beta),
        "durbin_watson": durbin_watson,
        "lag1_residual_autocorrelation": lag1,
        "autocorrelation_flag": autocorrelation_flag,
        "ar1_sensitivity_slope_per_year": ar1_slope,
        "ar1_sensitivity_p_value": ar1_p_value,
        "quadratic_adj_r_squared": quadratic_adj_r_squared,
        "quadratic_delta_adj_r_squared": quadratic_delta,
    }


trend_rows = []
for group_values, frame in annual_valence.groupby(["analysis_unit", "frame_stratum", "term_role", "target_group"], sort=True):
    analysis_unit, frame_stratum, term_role, target_group = group_values
    trend_rows.append(
        {
            "analysis_unit": analysis_unit,
            "frame_stratum": frame_stratum,
            "term_role": term_role,
            "target_group": target_group,
            "index_name": "valence_mean",
            **fit_trend(frame, "valence_mean"),
        }
    )
trend_summary = pd.DataFrame(trend_rows)
trend_summary.to_csv(TREND_SUMMARY_PATH, index=False)
trend_summary.head(12)


,analysis_unit,frame_stratum,term_role,target_group,index_name,n_years,year_center,linear_intercept,linear_slope_per_year,linear_slope_se,linear_p_value,linear_r_squared,linear_adj_r_squared,standardized_beta_year,durbin_watson,lag1_residual_autocorrelation,autocorrelation_flag,ar1_sensitivity_slope_per_year,ar1_sensitivity_p_value,quadratic_adj_r_squared,quadratic_delta_adj_r_squared
0,ADHD,clinical_only,target,ADHD,valence_mean,13,2020.0,0.051545,0.000427,0.000865,0.631173,0.021685,-0.067253,0.147257,1.219606,0.326709,True,0.001464,0.294530,0.536658,0.603911
1,ADHD,lived_only,target,ADHD,valence_mean,13,2020.0,0.115652,0.000342,0.000903,0.712171,0.012865,-0.076875,0.113424,2.693360,-0.760778,False,NaN,NaN,-0.071320,0.005555
2,ADHD,mixed,target,ADHD,valence_mean,13,2020.0,0.088108,0.002471,0.001201,0.064206,0.277784,0.212128,0.527052,1.869746,-0.111211,False,NaN,NaN,0.343562,0.131434
3,ADHD,substantive_core_overall,target,ADHD,valence_mean,13,2020.0,0.070123,0.001216,0.000674,0.098643,0.228324,0.158172,0.477833,1.323079,0.300468,False,NaN,NaN,0.431893,0.273722
4,Autism,clinical_only,target,Autism,valence_mean,13,2020.0,0.070912,0.001154,0.000904,0.228067,0.129021,0.049842,0.359196,1.175576,0.313860,True,0.002507,0.077325,0.655692,0.605850
5,Autism,lived_only,target,Autism,valence_mean,13,2020.0,0.153934,-0.001902,0.001220,0.147194,0.181041,0.106590,-0.425489,2.315609,-0.207332,False,NaN,NaN,0.149640,0.043050
6,Autism,mixed,target,Autism,valence_mean,13,2020.0,0.143105,0.000976,0.000621,0.144270,0.183427,0.109193,0.428284,1.280594,0.330018,False,NaN,NaN,0.449645,0.340452
7,Autism,substantive_core_overall,target,Autism,valence_mean,13,2020.0,0.110121,0.000571,0.000329,0.110393,0.215114,0.143761,0.463804,2.168136,-0.426358,False,NaN,NaN,0.236973,0.093212
8,frustration,unframed_baseline,baseline,baseline,valence_mean,13,2020.0,0.061893,0.002287,0.000789,0.014441,0.433308,0.381790,0.658261,0.399609,0.752701,True,0.007158,0.002728,0.889803,0.508013
9,loneliness,unframed_baseline,baseline,baseline,valence_mean,13,2020.0,0.027747,0.002453,0.000827,0.012791,0.444700,0.394219,0.666859,0.915264,0.330683,True,0.003889,0.009703,0.812832,0.418613


## Contributor Diagnostics

The top-collocate table identifies which NRC-VAD matches contribute most to positive or negative annual scores within each frame stratum. These diagnostics guide interpretation and help detect cases where a few words dominate a trajectory.


In [8]:
collocate_counts = (
    vad_matches.groupby(["analysis_unit", "frame_stratum", "lsc_year", "collocate", "collocate_type"], as_index=False)
    .agg(
        count=("collocate", "size"),
        valence=("valence", "mean"),
        arousal=("arousal", "mean"),
        dominance=("dominance", "mean"),
        documents=("doc_id", "nunique"),
    )
)
collocate_counts["weighted_valence_contribution"] = collocate_counts["count"] * collocate_counts["valence"]
collocate_counts["abs_weighted_valence_contribution"] = collocate_counts["weighted_valence_contribution"].abs()
collocate_counts["total_matches_for_unit_year_frame"] = collocate_counts.groupby(
    ["analysis_unit", "frame_stratum", "lsc_year"]
)["count"].transform("sum")
collocate_counts["match_share"] = collocate_counts["count"] / collocate_counts["total_matches_for_unit_year_frame"]

positive_top = (
    collocate_counts.sort_values(
        ["analysis_unit", "frame_stratum", "lsc_year", "weighted_valence_contribution"],
        ascending=[True, True, True, False],
    )
    .groupby(["analysis_unit", "frame_stratum", "lsc_year"])
    .head(10)
    .assign(contribution_direction="positive")
)
negative_top = (
    collocate_counts.sort_values(
        ["analysis_unit", "frame_stratum", "lsc_year", "weighted_valence_contribution"],
        ascending=[True, True, True, True],
    )
    .groupby(["analysis_unit", "frame_stratum", "lsc_year"])
    .head(10)
    .assign(contribution_direction="negative")
)
top_collocates = pd.concat([positive_top, negative_top], ignore_index=True).sort_values(
    ["analysis_unit", "frame_stratum", "lsc_year", "contribution_direction", "abs_weighted_valence_contribution"],
    ascending=[True, True, True, True, False],
)
top_collocates.to_csv(TOP_COLLOCATES_PATH, index=False)
top_collocates.head(20)


,analysis_unit,frame_stratum,lsc_year,collocate,collocate_type,count,valence,arousal,dominance,documents,weighted_valence_contribution,abs_weighted_valence_contribution,total_matches_for_unit_year_frame,match_share,contribution_direction
1430,ADHD,clinical_only,2014,disorder,unigram,84,-0.675000,0.530000,-0.494000,72,-56.700000,56.700000,7410,0.011336,negative
1431,ADHD,clinical_only,2014,depression,unigram,48,-0.938000,0.040000,-0.536000,48,-45.024000,45.024000,7410,0.006478,negative
1432,ADHD,clinical_only,2014,autism,unigram,80,-0.530000,0.000000,-0.106000,73,-42.400000,42.400000,7410,0.010796,negative
1433,ADHD,clinical_only,2014,diagnose,unigram,78,-0.455500,-0.035500,0.019500,68,-35.529000,35.529000,7410,0.010526,negative
1434,ADHD,clinical_only,2014,problem,unigram,37,-0.876000,0.288000,-0.170000,36,-32.412000,32.412000,7410,0.004993,negative
1435,ADHD,clinical_only,2014,drug,unigram,48,-0.665667,0.542000,-0.425000,43,-31.952000,31.952000,7410,0.006478,negative
1436,ADHD,clinical_only,2014,of,unigram,280,-0.111000,-0.119000,-0.161000,207,-31.080000,31.080000,7410,0.037787,negative
1437,ADHD,clinical_only,2014,anxiety,unigram,29,-0.708000,0.730000,-0.316000,29,-20.532000,20.532000,7410,0.003914,negative
1438,ADHD,clinical_only,2014,hyperactivity,unigram,26,-0.667000,1.000000,-0.333000,23,-17.342000,17.342000,7410,0.003509,negative
1439,ADHD,clinical_only,2014,disability,unigram,18,-0.854000,-0.036000,-0.654000,17,-15.372000,15.372000,7410,0.002429,negative


## Plots And Audit Flags

The report-facing trajectory figure uses three equal-width panels: ADHD, Autism, and comparator terms. The ADHD and Autism panels foreground the substantive-core Overall trajectory and add clinical/disorder and lived-experience traces as lighter contextual lines.

Mixed-frame estimates, coverage diagnostics, small-cell warnings, collocate concentration, and trend diagnostics remain available in the saved CSV tables and audit flags. They are not saved as separate report figures in order to keep the figure folder aligned with the main dissertation story.


In [ ]:
READER_FRAME_STRATA = ["clinical_only", "lived_only"]
READER_FRAME_LABELS = {
    "clinical_only": "Clinical/disorder framing",
    "lived_only": "Lived-experience framing",
}


def save_lsc_figure(fig: plt.Figure, png_path: Path) -> Path:
    fig.tight_layout(pad=1.1, rect=[0, 0, 1, 0.93])
    fig.savefig(png_path, dpi=LSC_FIGURE_DPI, bbox_inches="tight", facecolor="white")
    pdf_path = png_path.with_suffix(".pdf")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    return pdf_path


def series_color(unit: str, frame_stratum: str) -> str:
    if unit in CONDITION_FRAME_COLORS and frame_stratum in CONDITION_FRAME_COLORS[unit]:
        return CONDITION_FRAME_COLORS[unit][frame_stratum]
    if unit in LSC_UNIT_COLORS:
        return LSC_UNIT_COLORS[unit]
    return FRAME_COLORS.get(frame_stratum, "#7B8785")


def trend_line_for(series: pd.DataFrame, trend: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    years = series["lsc_year"].to_numpy(dtype=float)
    fitted = trend["linear_intercept"] + trend["linear_slope_per_year"] * (years - trend["year_center"])
    return years, fitted


def trend_for(unit: str, frame_stratum: str) -> pd.Series | None:
    row = trend_summary.loc[
        trend_summary["analysis_unit"].eq(unit) & trend_summary["frame_stratum"].eq(frame_stratum)
    ]
    if row.empty or pd.isna(row.iloc[0]["linear_slope_per_year"]):
        return None
    return row.iloc[0]


def y_limits_from(frame: pd.DataFrame, value_column: str, ci_low: str | None = None, ci_high: str | None = None) -> tuple[float, float]:
    values = [frame[value_column].to_numpy(dtype=float)]
    if ci_low and ci_low in frame:
        values.append(frame[ci_low].to_numpy(dtype=float))
    if ci_high and ci_high in frame:
        values.append(frame[ci_high].to_numpy(dtype=float))
    finite_values = [v[np.isfinite(v)] for v in values if len(v)]
    combined = np.concatenate(finite_values)
    low, high = float(combined.min()), float(combined.max())
    padding = max((high - low) * 0.10, 0.004)
    return low - padding, high + padding


def style_year_axis(ax: plt.Axes) -> None:
    ax.set_xticks(EXPECTED_YEARS[::2])
    ax.tick_params(axis="x", labelsize=8.5)


def plot_line_with_trend(
    ax: plt.Axes,
    frame: pd.DataFrame,
    unit: str,
    frame_stratum: str,
    value_column: str,
    color: str,
    marker: str,
    label: str | None = None,
    ci_low: str | None = None,
    ci_high: str | None = None,
    ribbon_alpha: float = 0.12,
    linewidth: float = 2.2,
    markersize: float = 4.8,
    alpha: float = 1.0,
    show_trend: bool = True,
) -> None:
    series = frame.loc[frame["analysis_unit"].eq(unit) & frame["frame_stratum"].eq(frame_stratum)].sort_values("lsc_year")
    if series.empty:
        return
    ax.plot(
        series["lsc_year"],
        series[value_column],
        marker=marker,
        markersize=markersize,
        linewidth=linewidth,
        label=label,
        color=color,
        alpha=alpha,
    )
    if ci_low and ci_high:
        ax.fill_between(
            series["lsc_year"].to_numpy(dtype=float),
            series[ci_low].to_numpy(dtype=float),
            series[ci_high].to_numpy(dtype=float),
            color=color,
            alpha=ribbon_alpha,
            linewidth=0,
        )
    trend = trend_for(unit, frame_stratum)
    if show_trend and trend is not None:
        years, fitted = trend_line_for(series, trend)
        ax.plot(years, fitted, color=color, linewidth=1.05, linestyle="--", alpha=min(alpha + 0.12, 0.92))


def plot_target_panel(ax: plt.Axes, unit: str) -> None:
    plot_line_with_trend(
        ax,
        annual_valence,
        unit,
        "substantive_core_overall",
        "valence_mean",
        series_color(unit, "substantive_core_overall"),
        FRAME_MARKERS["substantive_core_overall"],
        "Overall",
        "valence_ci_low",
        "valence_ci_high",
        ribbon_alpha=0.14,
        linewidth=2.8,
        markersize=4.8,
    )
    for frame_stratum in READER_FRAME_STRATA:
        plot_line_with_trend(
            ax,
            annual_valence,
            unit,
            frame_stratum,
            "valence_mean",
            series_color(unit, frame_stratum),
            FRAME_MARKERS[frame_stratum],
            READER_FRAME_LABELS[frame_stratum],
            "valence_ci_low",
            "valence_ci_high",
            ribbon_alpha=0.075,
            linewidth=1.55,
            markersize=3.7,
            alpha=0.82,
        )
    ax.set_title(unit, loc="left", fontsize=12, fontweight="bold")
    style_year_axis(ax)
    ax.legend(loc="best", fontsize=7.6)


def plot_baseline_panel(
    ax: plt.Axes,
    frame: pd.DataFrame,
    value_column: str,
    ci_low: str | None = None,
    ci_high: str | None = None,
    show_trend: bool = True,
) -> None:
    for unit in BASELINE_UNITS:
        plot_line_with_trend(
            ax,
            frame,
            unit,
            BASELINE_FRAME_STRATUM,
            value_column,
            LSC_UNIT_COLORS[unit],
            UNIT_MARKERS[unit] if "UNIT_MARKERS" in globals() else LSC_UNIT_MARKERS[unit],
            LSC_UNIT_LABELS[unit] if "LSC_UNIT_LABELS" in globals() else unit,
            ci_low,
            ci_high,
            ribbon_alpha=0.08,
            linewidth=2.0,
            show_trend=show_trend,
        )
    ax.set_title("Comparator terms", loc="left", fontsize=12, fontweight="bold")
    ax.legend(loc="best", fontsize=7.6)
    style_year_axis(ax)


main_rows = pd.concat(
    [
        annual_valence.loc[
            annual_valence["analysis_unit"].isin(TARGET_UNITS)
            & annual_valence["frame_stratum"].isin(["substantive_core_overall", *READER_FRAME_STRATA])
        ],
        annual_valence.loc[annual_valence["frame_stratum"].eq(BASELINE_FRAME_STRATUM)],
    ],
    ignore_index=True,
)
main_ylim = y_limits_from(main_rows, "valence_mean", "valence_ci_low", "valence_ci_high")

fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.35), sharex=True, sharey=True)
fig.suptitle("Sentiment: valence near target terms", fontsize=15, fontweight="bold", x=0.02, ha="left")
for ax, unit in zip(axes[:2], TARGET_UNITS):
    plot_target_panel(ax, unit)
plot_baseline_panel(
    axes[2],
    annual_valence.loc[annual_valence["frame_stratum"].eq(BASELINE_FRAME_STRATUM)],
    "valence_mean",
    "valence_ci_low",
    "valence_ci_high",
)
for ax in axes:
    ax.set_ylim(*main_ylim)
    ax.set_xlabel("Publication year")
axes[0].set_ylabel("Mean valence (-1 to 1)")
valence_pdf_path = save_lsc_figure(fig, VALENCE_PLOT_PATH)
plt.close(fig)

flags: list[dict[str, object]] = []
for _, row in coverage.iterrows():
    if row["matched_token_coverage"] < LOW_MATCHED_TOKEN_COVERAGE_WARN:
        flags.append(
            {
                "severity": "warn",
                "category": "low_matched_token_coverage",
                "lsc_year": int(row["lsc_year"]),
                "analysis_unit": row["analysis_unit"],
                "frame_stratum": row["frame_stratum"],
                "value": float(row["matched_token_coverage"]),
            }
        )
    if row["context_match_coverage"] < LOW_CONTEXT_COVERAGE_WARN:
        flags.append(
            {
                "severity": "warn",
                "category": "low_context_match_coverage",
                "lsc_year": int(row["lsc_year"]),
                "analysis_unit": row["analysis_unit"],
                "frame_stratum": row["frame_stratum"],
                "value": float(row["context_match_coverage"]),
            }
        )
    if bool(row.get("small_cell_flag", False)):
        flags.append(
            {
                "severity": "note",
                "category": "small_frame_year_cell",
                "lsc_year": int(row["lsc_year"]),
                "analysis_unit": row["analysis_unit"],
                "frame_stratum": row["frame_stratum"],
                "value": float(row["context_rows"]),
                "detail": f"documents={int(row['documents'])}",
            }
        )

for _, row in collocate_counts.loc[collocate_counts["match_share"] >= TOP_COLLOCATE_SHARE_WARN].iterrows():
    flags.append(
        {
            "severity": "warn",
            "category": "top_collocate_concentration",
            "lsc_year": int(row["lsc_year"]),
            "analysis_unit": row["analysis_unit"],
            "frame_stratum": row["frame_stratum"],
            "value": float(row["match_share"]),
            "detail": row["collocate"],
        }
    )

for _, row in trend_summary.loc[trend_summary["autocorrelation_flag"]].iterrows():
    flags.append(
        {
            "severity": "note",
            "category": "trend_residual_autocorrelation",
            "lsc_year": pd.NA,
            "analysis_unit": row["analysis_unit"],
            "frame_stratum": row["frame_stratum"],
            "value": float(row["durbin_watson"]),
            "detail": "AR(1) sensitivity slope reported in trend table",
        }
    )

audit_flags = pd.DataFrame(flags, columns=["severity", "category", "lsc_year", "analysis_unit", "frame_stratum", "value", "detail"])
audit_flags.to_csv(AUDIT_FLAGS_PATH, index=False)

print(f"Wrote {ANNUAL_VALENCE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {COVERAGE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {TOP_COLLOCATES_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {TREND_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {FRAME_CONTEXT_DIAGNOSTICS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {AUDIT_FLAGS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {VAD_MATCHES_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {VAD_CONTEXT_COVERAGE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {VALENCE_PLOT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {valence_pdf_path.relative_to(PROJECT_ROOT)}")
print(f"Audit warnings/notes: {len(audit_flags):,}")
annual_valence.sort_values(["analysis_unit", "frame_stratum", "lsc_year"]).head(20)


## Compact Handoff Summary

The handoff summary keeps the rerun easy to inspect: it shows the mean, sample standard deviation, and range of the annual valence estimates alongside coverage, warning counts, and trend slopes. The descriptive mean and standard deviation summarise annual trajectory values rather than collocate- or context-level observations.


In [10]:
handoff_summary = (
    annual_valence.groupby(["analysis_unit", "frame_stratum"], as_index=False)
    .agg(
        years=("lsc_year", "nunique"),
        valence_annual_mean=("valence_mean", "mean"),
        valence_annual_sd=("valence_mean", "std"),
        valence_min=("valence_mean", "min"),
        valence_max=("valence_mean", "max"),
        matched_units=("matched_vad_units", "sum"),
        documents_with_matches=("documents_with_matches", "sum"),
        min_matched_token_coverage=("matched_token_coverage", "min"),
        min_context_match_coverage=("context_match_coverage", "min"),
        small_cell_years=("small_cell_flag", "sum"),
    )
)
warning_counts = (
    audit_flags.groupby(["analysis_unit", "frame_stratum"]).size().rename("warnings").reset_index()
    if not audit_flags.empty
    else pd.DataFrame({"analysis_unit": [], "frame_stratum": [], "warnings": []})
)
handoff_summary = handoff_summary.merge(warning_counts, on=["analysis_unit", "frame_stratum"], how="left").fillna({"warnings": 0})
handoff_summary = handoff_summary.merge(
    trend_summary[["analysis_unit", "frame_stratum", "linear_slope_per_year", "linear_p_value", "autocorrelation_flag"]],
    on=["analysis_unit", "frame_stratum"],
    how="left",
)
handoff_summary["warnings"] = handoff_summary["warnings"].astype(int)
handoff_summary["small_cell_years"] = handoff_summary["small_cell_years"].astype(int)
handoff_summary


,analysis_unit,frame_stratum,years,valence_min,valence_max,matched_units,documents_with_matches,min_matched_token_coverage,min_context_match_coverage,small_cell_years,warnings,linear_slope_per_year,linear_p_value,autocorrelation_flag
0,ADHD,clinical_only,13,0.029206,0.066554,68768,7982,0.856495,0.981333,0,1,0.000427,0.631173,True
1,ADHD,lived_only,13,0.091842,0.140349,21835,2728,0.842701,0.981481,0,0,0.000342,0.712171,False
2,ADHD,mixed,13,0.054839,0.121359,12258,1627,0.839124,0.981818,1,1,0.002471,0.064206,False
3,ADHD,substantive_core_overall,13,0.053328,0.086189,102861,11420,0.858857,0.983407,0,0,0.001216,0.098643,False
4,Autism,clinical_only,13,0.054908,0.098382,128618,12955,0.846868,0.998168,0,1,0.001154,0.228067,True
5,Autism,lived_only,13,0.098980,0.167683,80459,9066,0.830529,0.997076,0,0,-0.001902,0.147194,False
6,Autism,mixed,13,0.130396,0.157774,40768,4734,0.835443,0.997312,0,0,0.000976,0.144270,False
7,Autism,substantive_core_overall,13,0.102910,0.122880,249845,24138,0.842999,0.998140,0,0,0.000571,0.110393,False
8,frustration,unframed_baseline,13,0.052208,0.095890,686724,93653,0.835410,0.998947,0,1,0.002287,0.014441,True
9,loneliness,unframed_baseline,13,0.016801,0.069383,243581,30431,0.845217,0.999417,0,1,0.002453,0.012791,True
